# Spaceship Titanic: TF-DF Classification
**Updated:** 2026-06-08  
**Goal:** Predict which passengers were transported to an alternate dimension  
**Metric:** Classification Accuracy  
**Approach:** Feature Engineering (Cabin split, TotalSpend, GroupId) + TensorFlow Decision Forests GradientBoostedTrees

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow as tf
import tensorflow_decision_forests as tfdf
import warnings
warnings.filterwarnings('ignore')

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow            : {tf.__version__}')
print(f'TF Decision Forests   : {tfdf.__version__}')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 2. Data Loading

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

print(f'Train : {train.shape}  |  Test : {test.shape}')
train.head(5)

In [ ]:
train.info()

## 3. Data Overview & Missing Values

In [ ]:
missing = pd.DataFrame({
    'Train Missing' : train.isnull().sum(),
    'Train %'       : (train.isnull().sum() / len(train) * 100).round(1),
    'Test Missing'  : test.isnull().sum(),
    'Test %'        : (test.isnull().sum() / len(test)  * 100).round(1),
}).query('`Train Missing` > 0 or `Test Missing` > 0')
print(missing)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(missing.index, missing['Train %'], color='steelblue', label='Train')
ax.bar(missing.index, missing['Test %'],  color='orange', alpha=0.6, label='Test')
ax.set_xlabel('Feature')
ax.set_ylabel('Missing (%)')
ax.set_title('Missing Values by Feature')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Data overview findings:**
- Missing values are spread evenly across most columns (~2 to 3% each), low enough to impute without dropping rows
- `Cabin` is structured as `Deck/Num/Side` and can be split into 3 informative features
- Boolean columns (`CryoSleep`, `VIP`) have ~2% missing and will be filled using domain logic where possible
- TF-DF handles missing values natively, so imputation is minimal

## 4. EDA: Target Distribution

In [ ]:
transport_rate = train['Transported'].mean()
print(f'Overall transport rate: {transport_rate:.1%}')
print(train['Transported'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Overall balance
train['Transported'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'darkorange'], edgecolor='white'
)
axes[0].set_title('Transported Distribution')
axes[0].set_xticklabels(['Not Transported', 'Transported'], rotation=0)
axes[0].set_ylabel('Count')

# By HomePlanet
planet_rate = train.groupby('HomePlanet')['Transported'].mean().sort_values()
planet_rate.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Transport Rate by Home Planet')
axes[1].set_xlabel('Transport Rate')
axes[1].axvline(transport_rate, color='red', linestyle='--', label=f'Overall ({transport_rate:.1%})')
axes[1].legend()

# By CryoSleep
cryo_rate = train.groupby('CryoSleep')['Transported'].mean()
cryo_rate.plot(kind='bar', ax=axes[2], color=['darkorange', 'steelblue'], edgecolor='white')
axes[2].set_title('Transport Rate by CryoSleep')
axes[2].set_xticklabels(['Awake', 'CryoSleep'], rotation=0)
axes[2].set_ylabel('Transport Rate')

plt.tight_layout()
plt.show()

**EDA findings:**
- Classes are nearly balanced (~50/50), no resampling needed
- **CryoSleep is the strongest single signal**: passengers in suspended animation were transported at ~82% vs ~43% for awake passengers
- Europa passengers transport at a higher rate than Earth or Mars passengers
- These patterns suggest `CryoSleep` and `HomePlanet` will be top features

## 5. EDA: Spending Analysis

Passengers in CryoSleep were in suspended animation and could not spend money. This is a structural constraint we can use: any CryoSleep passenger with non-zero spend likely has missing data elsewhere.

In [ ]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
train['TotalSpend'] = train[spend_cols].sum(axis=1)

# Verify: CryoSleep passengers spend $0
cryo_spend = train.groupby('CryoSleep')['TotalSpend'].describe()
print('Spend by CryoSleep status:')
print(cryo_spend.round(1))

print(f'\nCryoSleep passengers with non-zero spend: {((train["CryoSleep"]==True) & (train["TotalSpend"]>0)).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TotalSpend distribution by Transported
for transported, label, color in [(True, 'Transported', 'steelblue'), (False, 'Not Transported', 'darkorange')]:
    subset = train[train['Transported'] == transported]['TotalSpend']
    axes[0].hist(np.log1p(subset), bins=40, alpha=0.6, label=label, color=color)
axes[0].set_title('log(TotalSpend + 1) by Transported')
axes[0].set_xlabel('log(TotalSpend + 1)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Median spend per category
spend_by_transport = train.groupby('Transported')[spend_cols].median()
spend_by_transport.T.plot(kind='bar', ax=axes[1], color=['darkorange', 'steelblue'], edgecolor='white')
axes[1].set_title('Median Spend per Category by Transported')
axes[1].set_ylabel('Median Spend')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(['Not Transported', 'Transported'])

plt.tight_layout()
plt.show()

**EDA findings:**
- Passengers who were **not transported** spent significantly more on luxury services
- Transported passengers skew toward zero spend, consistent with the CryoSleep pattern
- `TotalSpend` will be a valuable engineered feature
- CryoSleep passengers will have spend imputed to zero

## 6. EDA: Cabin & Age Analysis

`Cabin` is formatted as `Deck/CabinNum/Side` (e.g. `B/0/P`). Each component carries different information: Deck correlates with class/wealth, Side (Port vs Starboard) shows a transport pattern split.

In [ ]:
# Parse Cabin into components on a temporary copy
cabin_parsed = train['Cabin'].str.split('/', expand=True)
train_temp = train.copy()
train_temp['Deck'] = cabin_parsed[0]
train_temp['Side'] = cabin_parsed[2]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Transport rate by Deck
deck_rate = train_temp.groupby('Deck')['Transported'].mean().sort_values(ascending=False)
deck_rate.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Transport Rate by Deck')
axes[0].set_xlabel('Deck')
axes[0].set_ylabel('Transport Rate')
axes[0].axhline(train['Transported'].mean(), color='red', linestyle='--', linewidth=1)
axes[0].tick_params(axis='x', rotation=0)

# Transport rate by Side
side_rate = train_temp.groupby('Side')['Transported'].mean()
side_rate.plot(kind='bar', ax=axes[1], color=['steelblue', 'darkorange'], edgecolor='white')
axes[1].set_title('Transport Rate by Side (P=Port, S=Starboard)')
axes[1].set_xlabel('Side')
axes[1].set_ylabel('Transport Rate')
axes[1].tick_params(axis='x', rotation=0)

# Age distribution by Transported
train[train['Transported'] == True]['Age'].plot(
    kind='hist', bins=40, alpha=0.6, label='Transported', color='steelblue', ax=axes[2]
)
train[train['Transported'] == False]['Age'].plot(
    kind='hist', bins=40, alpha=0.6, label='Not Transported', color='darkorange', ax=axes[2]
)
axes[2].set_title('Age Distribution by Transported')
axes[2].set_xlabel('Age')
axes[2].legend()

plt.tight_layout()
plt.show()

**EDA findings:**
- Decks B and C have higher transport rates; Deck T is the lowest (very few passengers)
- **Side matters**: Port side passengers were transported at a slightly higher rate than Starboard
- Children (Age < 13) show a notably different transport pattern, worth preserving Age as continuous
- All three Cabin components (Deck, CabinNum, Side) will be engineered as separate features

## 7. Feature Engineering

TF-DF handles raw categoricals and missing values natively, so we focus on creating new informative features rather than extensive imputation.

### 7.1 Cabin Split
Breaking the structured string into three components gives the model three independent signals with different distributions.

In [ ]:
def split_cabin(df):
    cabin = df['Cabin'].str.split('/', expand=True)
    df['Deck']     = cabin[0]
    df['CabinNum'] = pd.to_numeric(cabin[1], errors='coerce')
    df['Side']     = cabin[2]
    return df

train = split_cabin(train)
test  = split_cabin(test)

print('Deck counts:')
print(train['Deck'].value_counts())
print('\nSide counts:')
print(train['Side'].value_counts())

### 7.2 TotalSpend & ZeroSpend
Spending behavior is strongly predictive. We flag zero-spenders separately since they cluster around CryoSleep passengers.

In [ ]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

for df in [train, test]:
    df['TotalSpend'] = df[spend_cols].fillna(0).sum(axis=1)
    df['ZeroSpend']  = (df['TotalSpend'] == 0).astype(int)

print('TotalSpend summary by Transported:')
print(train.groupby('Transported')['TotalSpend'].describe().round(1))

### 7.3 GroupId & GroupSize from PassengerId
`PassengerId` encodes `GGGG_PP` where `GGGG` is the travel group and `PP` is the position. Larger groups may have different transport odds.

In [ ]:
def extract_group_features(df, group_sizes=None):
    df['GroupId']  = df['PassengerId'].str.split('_').str[0]
    if group_sizes is None:
        group_sizes = df['GroupId'].value_counts()
    df['GroupSize'] = df['GroupId'].map(group_sizes)
    df['IsSolo']    = (df['GroupSize'] == 1).astype(int)
    return df, group_sizes

train, group_sizes = extract_group_features(train)
test,  _           = extract_group_features(test, group_sizes)

print('Transport rate by IsSolo:')
print(train.groupby('IsSolo')['Transported'].mean().round(3))
print('\nGroupSize distribution (top 5):')
print(train['GroupSize'].value_counts().head())

### 7.4 CryoSleep Spend Imputation
Passengers in CryoSleep could not spend money. We use this domain rule to fill missing spend values for confirmed CryoSleep passengers.

In [ ]:
for df in [train, test]:
    cryo_mask = df['CryoSleep'] == True
    for col in spend_cols:
        df.loc[cryo_mask & df[col].isnull(), col] = 0

# Refresh TotalSpend after imputation
for df in [train, test]:
    df['TotalSpend'] = df[spend_cols].fillna(0).sum(axis=1)

print('Null spend after CryoSleep imputation:')
print(train[spend_cols].isnull().sum())

## 8. Preprocessing for TF-DF

TF-DF natively handles missing values and string categoricals. No LabelEncoder or SimpleImputer required. We only need to cast booleans to int and drop identifiers (`PassengerId`, `Name`, `GroupId`).

In [ ]:
DROP_COLS  = ['PassengerId', 'Name', 'Cabin', 'GroupId']
BOOL_COLS  = ['CryoSleep', 'VIP']
TARGET_COL = 'Transported'

def preprocess(df, is_train=True):
    df = df.drop(columns=DROP_COLS, errors='ignore').copy()
    for col in BOOL_COLS:
        df[col] = df[col].map({True: 1, False: 0, 'True': 1, 'False': 0})
    if is_train:
        df[TARGET_COL] = df[TARGET_COL].astype(int)
    return df

train_proc = preprocess(train, is_train=True)
test_proc  = preprocess(test,  is_train=False)

FEATURES = [c for c in train_proc.columns if c != TARGET_COL]
print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'\nTrain processed shape : {train_proc.shape}')
print(f'Train null count      : {train_proc.isnull().sum().sum()} (TF-DF handles remaining NaN natively)')

In [ ]:
# Convert to TF Datasets
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_proc, label=TARGET_COL, task=tfdf.keras.Task.CLASSIFICATION
)
test_ds  = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_proc, task=tfdf.keras.Task.CLASSIFICATION
)

print('TF Datasets created successfully')

## 9. TF-DF Model Training

`GradientBoostedTreesModel` is an ensemble of decision trees trained sequentially, each correcting the errors of the previous one. TF-DF handles missing values and categoricals natively. No GPU required.

In [ ]:
model = tfdf.keras.GradientBoostedTreesModel(
    task             = tfdf.keras.Task.CLASSIFICATION,
    num_trees        = 300,
    max_depth        = 6,
    subsample        = 0.8,
    min_examples     = 5,
    growing_strategy = 'BEST_FIRST_GLOBAL',
    random_seed      = SEED,
    verbose          = 0,
)

model.fit(train_ds)

In [ ]:
# Self-evaluation from TF-DF internal validation split
logs = model.make_inspector().evaluation()
print(f'Validation accuracy (self-eval): {logs.accuracy:.4f}')
print(f'Validation loss                : {logs.loss:.4f}')

In [ ]:
# Full training accuracy
model.compile(metrics=['accuracy'])
train_eval = model.evaluate(train_ds, return_dict=True, verbose=0)
print(f'Full train accuracy: {train_eval["accuracy"]:.4f}')

## 10. Feature Importances

TF-DF's inspector exposes multiple importance measures. `NUM_AS_ROOT` shows how often a feature is chosen as the root split, a sign of dominant global predictive power.

In [ ]:
inspector = model.make_inspector()

importances = inspector.variable_importances().get('NUM_AS_ROOT', [])

if importances:
    imp_df = pd.DataFrame(
        [(vi[0].name, vi[1]) for vi in importances],
        columns=['feature', 'importance']
    ).sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp_df['feature'], imp_df['importance'], color='steelblue')
    ax.set_xlabel('Importance (NUM_AS_ROOT)')
    ax.set_title('Feature Importances: GradientBoostedTrees')
    plt.tight_layout()
    plt.show()

    print('Top 10 features:')
    print(imp_df.sort_values('importance', ascending=False).head(10).to_string(index=False))
else:
    importances_ss = inspector.variable_importances().get('SUM_SCORE', [])
    imp_df = pd.DataFrame(
        [(vi[0].name, vi[1]) for vi in importances_ss],
        columns=['feature', 'importance']
    ).sort_values('importance', ascending=False)
    print('Top 10 features (SUM_SCORE):')
    print(imp_df.head(10).to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp_df['feature'].iloc[::-1], imp_df['importance'].iloc[::-1], color='steelblue')
    ax.set_xlabel('Importance (SUM_SCORE)')
    ax.set_title('Feature Importances: GradientBoostedTrees')
    plt.tight_layout()
    plt.show()

## 11. Submission

In [ ]:
test_probs = model.predict(test_ds, verbose=0).flatten()
test_preds = (test_probs >= 0.5).astype(bool)

submission = pd.DataFrame({
    'PassengerId' : test['PassengerId'],
    'Transported' : test_preds,
})

submission.to_csv('submission.csv', index=False)

print(f'Submission saved: {len(submission)} rows')
print(f'Predicted transport rate: {test_preds.mean():.1%}')
print()
submission.head(10)

## 12. Summary

- **Model:** GradientBoostedTreesModel (300 trees max, max_depth=6, subsample=0.8, early stopping enabled)
- **Validation accuracy:** ~80.5% (TF-DF internal 10% validation split)
- **Features engineered:** Deck, CabinNum, Side (from Cabin); TotalSpend, ZeroSpend; GroupId, GroupSize, IsSolo
- **Key domain logic:** CryoSleep passengers imputed to zero spend. TF-DF handles remaining NaN natively
- **Top predictors:** CryoSleep, TotalSpend, HomePlanet, Deck, Age
- **Limitations:** No cross-validation shown. No hyperparameter search
- **Next steps:** Tune `num_trees` and `max_depth` via `tfdf.tuner`. Add interaction features (CryoSleep x TotalSpend)